<div style="background: linear-gradient(135deg, #0a0a0a 0%, #1a1a1a 100%); padding: 40px 48px; border-left: 5px solid #C9A84C; border-radius: 4px; margin-bottom: 8px;">
  <h1 style="color: #C9A84C; font-family: 'Segoe UI', sans-serif; font-size: 2.4rem; margin: 0 0 8px 0; letter-spacing: 1px;">
    TaxiPilotFE
  </h1>
  <h2 style="color: #f0ece4; font-family: 'Segoe UI', sans-serif; font-size: 1.2rem; font-weight: 300; margin: 0 0 16px 0;">
    Angular 16 → React 18 Migration
  </h2>
  <p style="color: #b0a89a; margin: 0; font-size: 0.95rem;">
    📅 Session date: 16 May 2026 &nbsp;|&nbsp; 👤 For: Abdullah
  </p>
</div>

<div style="background: #f9f6f0; border-left: 4px solid #C9A84C; padding: 24px 32px; border-radius: 2px; margin: 16px 0;">
  <h2 style="color: #0a0a0a; font-family: 'Segoe UI', sans-serif; margin: 0 0 12px 0; font-size: 1.3rem;">
    🗺️ The Big Picture
  </h2>
  <p style="color: #3d3a3a; font-size: 1rem; line-height: 1.7; margin: 0;">
    The entire frontend was converted from <strong>Angular 16</strong> to <strong>React 18 + Vite</strong> — same visual design, different technology underneath. The old Angular code is safely preserved in <code style="background:#eee; padding: 2px 6px; border-radius: 3px;">angular-archive/</code> if you ever need to look back at it.
  </p>
</div>

| Layer | Angular (old) | React (new) |
|---|---|---|
| Framework | Angular 16 | React 18 + Vite 5 |
| State | NgRx | Zustand |
| Routing | Angular Router | React Router v6 |
| HTTP | HttpClient + RxJS | native `fetch` |
| Styles | SCSS modules | Plain CSS (custom properties) |
| Maps | `@angular/google-maps` | native `window.google` SDK |
| Real-time | `@microsoft/signalr` | `@microsoft/signalr` (same) |

<div style="background: #0a0a0a; padding: 20px 32px; border-radius: 4px; margin: 16px 0 8px 0;">
  <h2 style="color: #C9A84C; font-family: 'Segoe UI', sans-serif; margin: 0; font-size: 1.3rem; letter-spacing: 0.5px;">
    📦 The 5 Commits
  </h2>
</div>

Each commit was kept focused on one concern so the history is easy to read.

| # | Commit title | What's inside |
|:---:|---|---|
| **1** | Archive Angular | Moved all old Angular files into `angular-archive/`, deleted Angular config files |
| **2** | Build config | `vite.config.ts`, `tsconfig.json`, `package.json`, `index.html` — the project setup |
| **3** | Data layer | `src/models/`, `src/store/`, `src/services/` — how data is fetched and shared |
| **4** | Components | `src/components/` — Navbar, MapComponent, QuoteAirport, Footer, Toast, Cookie … |
| **5** | Pages & styles | `src/pages/`, `src/index.css`, `public/` — the actual screens and global CSS |

### New folder structure at a glance

```
src/
├── main.tsx              ← app entry point
├── App.tsx               ← routes
├── index.css             ← all global styles (replaces SCSS)
├── constants/            ← API keys, base URLs
├── models/               ← TypeScript interfaces (GoogleMapInfo, UserLogin, Visitor)
├── store/                ← Zustand global state (appStore.ts)
├── services/             ← api.ts, signalr.ts, localStorage.ts
├── components/           ← reusable UI pieces
│   ├── sections/         ← OurService, WhyChooseUs (home page blocks)
│   └── ...
├── pages/                ← one file per route (Home, About, Contact …)
└── types/                ← global.d.ts (window.google type)

public/                   ← static assets (images, icons) — served as-is by Vite
angular-archive/          ← original Angular source, read-only reference
```

<div style="background: #0a0a0a; padding: 20px 32px; border-radius: 4px; margin: 16px 0 8px 0;">
  <h2 style="color: #C9A84C; font-family: 'Segoe UI', sans-serif; margin: 0; font-size: 1.3rem; letter-spacing: 0.5px;">
    🔧 Problems Fixed Along the Way
  </h2>
</div>

---

### 1 · Visual Design
**Problem:** The React version came out with a plain amber navbar and a totally different hero layout.  
**Fix:** Matched every style back to the original Angular SCSS:
- Navbar → black `#0a0a0a` background, gold `#C9A84C` hamburger and links
- Hero → rotating background images (3 photos, cycling every 20 s), glass panel with `backdrop-filter: blur`, gold left border `border-left: 3px solid #C9A84C`
- Sections like *Customer Say*, *Our Commitment*, and the CTA block were hidden in the original — kept them hidden in React too

---

### 2 · Quote / Booking Window Layout
**Problem:** The booking form was rendered as a small popup modal.  
**Fix:** Rebuilt as a full-screen layout — the form panel sits on the left (`min-width: 396px`, `position: absolute`, `z-index: 1`) and the Google Map fills the entire screen behind it (`z-index: 0`), exactly matching the original.

---

### 3 · Google Maps Route
**Problem:** The right panel showed a grey placeholder, no map.  
**Fix:** Created `MapComponent.tsx` using the native `window.google.maps` SDK:
- `DirectionsService` calculates the driving route
- `DirectionsRenderer` draws it in **purple** (`#800080`, weight 6)
- Picks the shortest route if multiple alternatives exist
- Calls `fitBounds` so both markers are visible, capped at zoom 12

---

### 4 · Get Fare Button — 3 stacked bugs

| Bug | Root cause | Fix |
|---|---|---|
| **URL encoding** | `encodeURIComponent()` was turning `09:00` into `09%3A00`, which the backend didn't recognise | Changed to only encode spaces (`%20`), leaving colons and other chars as-is |
| **CORS** | `fetch` was calling the full external URL directly; browsers block cross-origin requests | Switched all API calls to relative paths (`/api/...`) so the Vite proxy handles the request server-to-server |
| **Expired SSL cert** | The backend's HTTPS certificate had expired; the proxy refused to connect | Added `secure: false` to the Vite proxy config to bypass cert validation in dev |

---

### 5 · Google Maps API Key
**Problem:** Maps showed *"This page didn't load Google Maps correctly"* on localhost.  
**Fix (you did this):** Added `localhost:4503/*` to the allowed HTTP referrers for the API key in Google Cloud Console. The `async defer` was also removed from the script tag to match how Angular loaded it (synchronously), which fixed the autocomplete timing issue.

<div style="background: linear-gradient(135deg, #0a0a0a 0%, #1a1a1a 100%); border: 1px solid rgba(201,168,76,0.3); padding: 24px 32px; border-radius: 4px; margin-top: 16px;">
  <h3 style="color: #C9A84C; font-family: 'Segoe UI', sans-serif; margin: 0 0 12px 0;">⚡ Quick Reference — Dev Commands</h3>
  <table style="color: #f0ece4; font-family: monospace; border-collapse: collapse; width: 100%;">
    <tr>
      <td style="padding: 6px 16px 6px 0; color: #b0a89a;">Start dev server</td>
      <td style="padding: 6px 0;"><code style="background: rgba(201,168,76,0.15); padding: 3px 10px; border-radius: 3px; color: #C9A84C;">npm run dev</code></td>
      <td style="padding: 6px 0 6px 16px; color: #b0a89a;">→ http://localhost:4503</td>
    </tr>
    <tr>
      <td style="padding: 6px 16px 6px 0; color: #b0a89a;">Production build</td>
      <td style="padding: 6px 0;"><code style="background: rgba(201,168,76,0.15); padding: 3px 10px; border-radius: 3px; color: #C9A84C;">npm run build</code></td>
      <td style="padding: 6px 0 6px 16px; color: #b0a89a;">→ outputs to dist/</td>
    </tr>
    <tr>
      <td style="padding: 6px 16px 6px 0; color: #b0a89a;">Type check only</td>
      <td style="padding: 6px 0;"><code style="background: rgba(201,168,76,0.15); padding: 3px 10px; border-radius: 3px; color: #C9A84C;">npx tsc --noEmit</code></td>
      <td style="padding: 6px 0 6px 16px; color: #b0a89a;">no output = all good</td>
    </tr>
  </table>
</div>